In [0]:
# Copy the TEST champion into the PROD catalog and promote it to PROD `champion`
# only when its validation RMSE beats the current PROD champion by at least `min_improvement`.
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from mlflow import MlflowClient
from utils import (load_project, model_name as qualified_model_name,
                   classifier_model_name as qualified_classifier_name, widget_or)

mlflow.set_registry_uri("databricks-uc")
cfg = load_project()

dbutils.widgets.text("source_catalog", "stock_demo_test")
dbutils.widgets.text("target_catalog", "stock_demo_prod")
dbutils.widgets.text("min_improvement", "")
source_catalog = dbutils.widgets.get("source_catalog")
target_catalog = dbutils.widgets.get("target_catalog")
min_improvement = float(widget_or(dbutils.widgets.get("min_improvement"),
                                  cfg["model"]["min_validation_rmse_improvement"]))

source_model_name = qualified_model_name(cfg, source_catalog)
target_model_name = qualified_model_name(cfg, target_catalog)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.models")

client = MlflowClient()

# --------------------------------------------------
# Source environment's current champion
# --------------------------------------------------
source_champion = client.get_model_version_by_alias(source_model_name, "champion")
source_run = client.get_run(source_champion.run_id)
source_metric = float(source_run.data.metrics["val_rmse"])
print(f"Source champion: {source_model_name} v{source_champion.version} (val_rmse={source_metric:.6f})")

# --------------------------------------------------
# Copy the model version across catalogs
# --------------------------------------------------
copied_mv = client.copy_model_version(
    src_model_uri=f"models:/{source_model_name}/{source_champion.version}",
    dst_name=target_model_name,
)
print(f"Copied into {target_model_name} as version {copied_mv.version}")

# --------------------------------------------------
# Compare against PROD's existing champion (if any) before promoting
# --------------------------------------------------
try:
    prod_champion = client.get_model_version_by_alias(target_model_name, "champion")
    prod_metric = float(client.get_run(prod_champion.run_id).data.metrics["val_rmse"])
    print(f"Current prod champion: v{prod_champion.version} (val_rmse={prod_metric:.6f})")
except Exception:
    prod_metric = float("inf")
    print("No existing prod champion found - this will become champion by default.")

promoted = source_metric < prod_metric * (1.0 - min_improvement)
if promoted:
    client.set_registered_model_alias(target_model_name, "champion", str(copied_mv.version))
    print(f"Promoted: prod 'champion' now points to version {copied_mv.version}")
else:
    client.set_registered_model_alias(target_model_name, "challenger", str(copied_mv.version))
    print("Not promoted - copied version registered as 'challenger' only.")

dbutils.jobs.taskValues.set(key="promoted", value=bool(promoted))


# Copy the independent direction-classification champion to the target catalog.
source_classifier_name = qualified_classifier_name(cfg, source_catalog)
target_classifier_name = qualified_classifier_name(cfg, target_catalog)
source_classifier = client.get_model_version_by_alias(source_classifier_name, "champion")
source_classifier_run = client.get_run(source_classifier.run_id)
source_classifier_score = float(source_classifier_run.data.metrics["val_balanced_accuracy"])
classifier_copy = client.copy_model_version(
    src_model_uri=f"models:/{source_classifier_name}/{source_classifier.version}",
    dst_name=target_classifier_name,
)
try:
    target_classifier_champion = client.get_model_version_by_alias(target_classifier_name, "champion")
    target_classifier_score = float(client.get_run(target_classifier_champion.run_id).data.metrics["val_balanced_accuracy"])
except Exception:
    target_classifier_score = float("-inf")
classifier_promoted = source_classifier_score > target_classifier_score
if classifier_promoted:
    client.set_registered_model_alias(target_classifier_name, "champion", str(classifier_copy.version))
else:
    client.set_registered_model_alias(target_classifier_name, "challenger", str(classifier_copy.version))
print(f"Classifier copy: {target_classifier_name} v{classifier_copy.version}; "
      f"source validation balanced accuracy={source_classifier_score:.4f}; promoted={classifier_promoted}")